In [3]:
import pandas as pd
import re
from pathlib import Path

In [4]:
def load_ambr_csv(csv_file: str | Path) -> pd.DataFrame:
    df = pd.read_csv(csv_file)
    # Normalize time column name
    time_candidates = ["time", "Time", "Time_min", "t", "t_min", "minutes"]
    tcol = next((c for c in time_candidates if c in df.columns), None)
    if tcol is None:
        raise KeyError(f"No time column among {time_candidates}. Found: {list(df.columns)}")
    df = df.rename(columns={tcol: "time"})
    # Ensure starts at 0 (minutes)
    t0 = float(df["time"].min())
    df["time"] = df["time"] - t0
    return df.sort_values("time").reset_index(drop=True)


csv_path = Path("./experimental_dataset/ambr_run1_140323_19-24.csv")
df_raw = load_ambr_csv(csv_path)
df_raw.head()
#--------------------------------------------------------------------------------

# detecting bioreactors in the already-loaded file
def detect_bioreactor_ids(columns):
    ids = set()
    pat = re.compile(r"^Bioreactor\s+(\d+)\s+-\s+")
    for c in columns:
        m = pat.match(c)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

# visualize what are the bioreactors available in the file
rids = detect_bioreactor_ids(df_raw.columns)
if not rids:
    raise RuntimeError("No 'Bioreactor <ID> - <Signal>' columns found. Check CSV headers.")
print("Detected reactor IDs:", rids)

# selecting the bioreactor
reactor_id = 19

Detected reactor IDs: [19, 21, 22, 23, 24]


In [12]:
bioreactor_19_cols = [c for c in df_raw.columns if c.startswith(f"Bioreactor {reactor_id} -")]
df_19 = df_raw[["time"] + bioreactor_19_cols]
df_19.columns

Index(['time', 'Bioreactor 19 - Acid volume pumped',
       'Bioreactor 19 - Air flow', 'Bioreactor 19 - Base volume pumped',
       'Bioreactor 19 - Bioreactor pressure reading', 'Bioreactor 19 - CER',
       'Bioreactor 19 - DO', 'Bioreactor 19 - Feed#1 flow rate',
       'Bioreactor 19 - Feed#1 volume pumped',
       'Bioreactor 19 - Feed#2 flow rate',
       'Bioreactor 19 - Feed#2 volume pumped', 'Bioreactor 19 - Foam sensor',
       'Bioreactor 19 - Off-gas CO2%', 'Bioreactor 19 - Optical density',
       'Bioreactor 19 - OUR', 'Bioreactor 19 - pH',
       'Bioreactor 19 - Reflectance', 'Bioreactor 19 - RQ',
       'Bioreactor 19 - Sampling events', 'Bioreactor 19 - Stir speed',
       'Bioreactor 19 - Temperature', 'Bioreactor 19 - Volume'],
      dtype='object')